# Redundancy Detection with Dleto Chiseling

`CC-BY 2025 Brooksbank, Kassabov, Wilson`

This tutorial uses Dleto's "Tucker Chisels" to reproduce a Tucker Decompositions, also known as radical/total zero-divisor detection.

> A **Tucker Decomposition** of a tensor $\Gamma$ framed by axes (a.k.a. modes/legs/indices) $\mathbb{K}^{d_1},\ldots, \mathbb{K}^{d_{\ell}}$, is a subset of axes $A\subset\{1,\ldots,\ell\}$ and a decomposition
> $$\forall a\in A,\qquad \mathbb{K}^{d_a}=E_a\oplus R_a$$
> such that $\Gamma$ contracted on $R_a$ is 0.  

The decompositions can be given by a partitioned bases of $E_a$ and $R_a$, or equivalently by a linear projections $e_a:\mathbb{K}^{d_a}\to \mathbb{K}^{e_a}$ with kernel $R_a$.  In many situations the purpose of a Tucker decomposition is to restrict the tensor the $E_a$-spaces.  In that case it is sufficient to return only $E_a$ (respectively $e_a$) and to refer to that restriction as the Tucker Decomposition.  In this tutorial however the term "Tucker Decomposition" will refer to the complete decomposition.

 1. [Loading Dleto](#1-loading-dleto)
 2. [Creating a Tensor Experiment](#2-creating-tensor-experiment)
 3. [Tucker chiseling](#3-tucker-chiseling)
 4. Derivation selection.
___

**Performance Remark.** Tucker decompositions have been explored since the 1800's and as such there are many optimized strategies to discover them.  This tutorial therefore should be seen as using a familiar problem to explore the range of options of Dleto chisels and the use of the parameters available to Dleto chiseling.  Unfortunately, Dleto chiselling operates with a complexity slightly greater than many more direct strategies for Tucker decompositions.  For high-performance computations, Dleto automatically switches to alternative optimized strategies by calling `nondeg`. 


## 1. Loading Dleto

Start by loading `Dleto.jl`.  If this is your first time you may need to install auxiliary packages and possibly set up Julia for notebooks.  That is a one-time setup for most users, see instructions here or consider using the fully online Binder demonstration.

In [ ]:
# Uncomment and run the first time, if Dleto is not installed
# using Pkg

# Option 1: To install from remote repository, use:
# Pkg.add(url="https://github.com/thetensor-space/OpenDleto")

# Option 2: If cloned locally at PATH 
# Pkg.activate( PATH ) 

If you have already added Dleto to your Julia packages begin by loading the necessary packages, `ITensors` for general tensor controls, `Plots` for visualization tools, and `Dleto` the primary package of chisel techniques.

In [ ]:
using ITensors
using Plots
using Dleto

## 2 Creating Tensor Experiment

Our first experiment is the simplest demonstration of chiseling uncovering some form of structure.  

We create two tensors, one randomized for control, and an identical in size but with 2 rows, columns, and slices set to all zero.  We then randomize the experiment tensor by applying a change in coordinates.  The goal is to using chiseling to detect and recover the hidden zero rows/columns/slices.

We note that this experiment is extremely basic and there are faster algorithms to recover this structure known under the names of **detecting radicals** or **Tucker decompositions**.  `Delto.jl` makes use of those faster methods for large scale experiments, but this experiment simply uses generic Delto methods to explore what is possible.


> **Note** Julia being mathematically oriented accepts $\LaTeX$ styled commands with tab-completion.  For example to insert the Unicode character for $\Gamma$ use `\Gamma` in the code area followed by `tab` (or cut-and-paste a character you see somewhere else).  Or replace with an simpler string of characters of your liking.  One suggestion, `Dleto.jl` calculations make substantial use of tensors, matrices, and lists of matrices.  A convention that clearly indicates those roles will be an investment worth your time.  
>
> We will be using:
> * capitol Greek letters `Γ` (`\Gamma`), `Δ` (`\Delta`), `Σ` (`\Sigma`), `Ξ` (`\Xi`), `Υ` (`\Upsilon`), etc. for tensors
> * capital English letters `X`, `Y`, `Z` etc. for matrices
> * lower case Geek letters for real numbers
> * lower case English letters for integers
> * Plural for lists of data, for instance `Γs` and `Xs`.

In [ ]:
ds = (5,5,5); rs=(3,3,3)
Γ = randn(Float64, ds);  # a control tensor

# the experiment tensor is initially 0 but we fill in a subregion.
Δ = zeros(Float64, ds);  
Δ[1:rs[1], 1:rs[2], 1:rs[3]] .= randn(rs...)

side_by_side(Γ, Δ; left_title="Control Γ", right_title="Experiment Δ")

For larger tensors it may help to do a visualization instead, and we can do this with the following.

In [ ]:
p1 = plot_tensor(Γ; title="Control Γ", color=:blue)
p2 = plot_tensor(Δ; title="Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

The control tensor very likely has few to no all zero axes whereas the experiment evidently does.  The role of chiseling is to recover such anamolies without knowing ahead of time.  So we now randomize the experiment so as to obscure this data.  Note that the randomization applies random bases change so the original coordinates.  

It should now be much less obvious that the control and experminet are any different.

In [ ]:
# repeat all the conversions on control, makes sure types stay in sync.
Γ_rand, Xs = randomize_tensor(Γ);
isapprox(Γ * Xs, Γ_rand)

Δ_rand, Ys = randomize_tensor(Δ);  # In theory Δ_rand = Δ*Xs
# Equality check is often too strict, approximate check is better
isapprox(Δ * Ys, Δ_rand)

You may discover `randomize` has switched the internal representation of tensors the `ITensor` format.  In fact this why we are able to apply a **list** (Julia `Vector` types) of matrices to `Δ` without specifying what side to multiply on.  The order is immaterial, for example `Xs*Δ` is allowable as well.  This is in contrast to matrix multiplication where sides indicate the axis of application.  `ITensors` uses internal markers to permit the correct application across multiple axes.

To see the contents in its original array format you can use `asarray(Δ)`.

In [ ]:
side_by_side(Γ_rand, Δ_rand;
            left_title="Control Γ", right_title="Experiment Δ_rand")

Or see it again as a plot.

In [ ]:
p1 = plot_tensor(Γ_rand; title="Randomized Control Γ", color=:blue)
p2 = plot_tensor(Δ_rand; title="Randomized Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

# 3. Tucker Chiseling

While the command `nondeg` performs the Tucker decomposition on one step, it does so with internal optimizations that mostly do not relate to the chisel strategy more generally.  So to break-down the Tucker decomposition using Tucker Chisels specifically we make a 3 step process.
 * Select the appropriate Tucker chisel
 * Compute derivations of the tensor with the Tucker chisel.
 * Use the derivations to stratify the original tensor.

First let us use the default universal chisel to see what happens.

In [ ]:
Γ_strat, Xs = stratify(Γ_rand);
Δ_strat, Ys = stratify(Δ_rand);

Now let us plot the results of generic stratification.  In a typical situation the control tensor will remain random scatter plot of values spread across all axes.  Meanwhile the stratified experiment will recover a cluster surrounded by nearly zero values.  The precise position of the nonzeros can vary but the total nonzero cluster should be of dimensions `ds-rs`.

In [ ]:
p1 = plot_tensor(Γ_strat; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_strat; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

The plots here are showing dots whose volume is proportional to scalar size, which can omit tiny values.  Inspecting the actual data tells the store in more detail.  In particular we do not manage to up a small tolerance we see that we have recovered the degeneracy planted in the experimental tensor.

In [ ]:
side_by_side(Γ_strat, Δ_strat;
            left_title="Stratified Control Γ", right_title="Stratified Experiment Δ")

# 3. Larger scale

Now let us increase the scale.

In [ ]:
ds = (50,50,50); rs=(45,47,49)
Γ = randn(Float64, ds);  # a control tensor
Γ_rand, Xs = randomize_tensor(Γ);

# the experiment tensor is initially 0 but we fill in a subregion.
Δ = zeros(Float64, ds);  
Δ[1:rs[1], 1:rs[2], 1:rs[3]] .= randn(rs...)
Δ_rand, Ys = randomize_tensor(Δ);  # In theory Δ_rand = Δ*Xs

p1 = plot_tensor(Γ_rand; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_rand; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

Now lets us begin to chisel.

In [ ]:
Γ_strat, Xs = stratify(Γ_rand);
Δ_strat, Ys = stratify(Δ_rand);


In [ ]:

p1 = plot_tensor(Γ_strat; title="Stratified Control Γ", color=:blue)
p2 = plot_tensor(Δ_strat, 1e-14; title="Stratified Experiment Δ", color=:red)
Plots.plot(p1, p2, layout=(1,2), size=(1000, 400))

In [ ]:
as = Array(Δ_strat,inds(Δ_strat)...);
heatmap(as[:,:,25]', title="Slice of Δ_rand at index 25", color=:reds)
# as[:,25,:]

# 4. 

You see the default setting used only 10 of the available 75 values.  The value $75=3\cdot 5^2$ coincides with being a 3-valent tensor with dimensions `(5,5,5)` so it will take three $5\times 5$-operators to manipulate for a total of 75 parameters.  However there are also $125=5^3$ constraint equations so we should expect few solutions.  Thus as a practical improvement, `Dleto.jl` estimates only 10 solutions will be necessary and accelorate the calculations knowing this heuristic value.

To explain the process in full, let us override this heuristic cut-off and take all the available values.  Note, if we give `chisel` a number greater than the available values or one that is negative, then all values are computed. 

The behavior demonstrated is a response to the number of 0's we included as rows, columns, and slices in the experiment.  It may become more noticeable to plot the derviatives, using `diff(xxx_t.values)`.  

Unfortunately, the steep changes in behavior near the final values can overwhellem the comparisons we are after near the small values, which is what we care to track.  Those carry little useable information as we can see they occur at the same range.  It is for this reason that we typically do not include the larger values.  By replotting the data with smaller range we can better see the differences between control and experiment.

**We are interested in the index location of the first spike in the differences.**

In [ ]:
p1 = plot(diff(ctr_spall.values[1:50]),  xlabel="Index", ylabel="Value", title="Control Spall Values", color=:blue)
p2 = plot(diff(exp_spall.values[1:50]),  xlabel="Index", ylabel="Value", title="Experiment Spall Values", color=:red)
plot(p1, p2, layout=(1,2), size=(1000, 400))

You may notice the number of 0's in the experimental plot increase with the number of all zero rows, columns, and slices. You may even be able to estimate a formula on the order of the following.
```math
2 + d_1\cdot r_1 + d_2 \cdot r_2 + d_3\cdot r_3
```
> While not important to chisel selection, the orgin of this formula can be seen as the number of matrices mapping $\mathbb{R}^{d_i}\to \mathbb{R}^{r_i}$ for each axis $i\in \{1,2,3\}$ plus the initial 2 dimensions that are present for all tensors of valence 2.  Note that these numbers are also specific to the universal chisel which we get by default when using `chisel(xxx_t)`.

## Recovering blocks from the chiseling spall


In [ ]:
sculpted_ctr = sculpt(ctr_t, ctr_spall, collect(3:3));
sculpted_exp = sculpt(exp_t, exp_spall, collect(3:3));
sidebyside(round.(100*sculpted_ctr.tensor), round.(100*sculpted_exp.tensor); 
left_title="Control Sculpted Vec", right_title="Experiment Sculpted Vec")